> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Key concepts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/main/docs/user_guide/01_key_concepts.ipynb)

Your code computes an answer from its inputs.
When one of those inputs is uncertain, the answer is uncertain too, and plugging in a best guess for the input can give the wrong answer.
This chapter introduces the eight concepts that ProbPipe uses to compute with uncertain inputs.
It then applies them to one example: how far a car travels while braking when its speed is known only roughly.
The chapter assumes that you know NumPy or JAX and basic probability.

## The concepts

### What goes into a computation?

Three kinds of object, called **terms**, go into a computation and come out of it:

- **Value:** something you know, such as a measured length. `NumericArray(jnp.array([4.0, 5.0]))` holds numeric data, and `.raw` gives it back as a plain array.
- **Distribution:** something you are unsure about, described by how likely each of its possible values is. `Normal("speed", 25.0, 4.0)` says that the speed is about 25, give or take 4, and its first argument names the quantity.
- **Function:** your own Python code, written as if every input were known. Decorating it with `@function` lets it accept distributions as well as values.

### What happens when an input is uncertain?

**Lifting.** Pass a distribution where your function expects a number, and the call returns the distribution of the output.
The result holds every output that the uncertainty allows, each weighted by how likely it is, and you do not rewrite the function.
To compute it, ProbPipe draws 256 inputs, runs your function on each one, and returns the 256 outputs as an `EmpiricalDistribution`.

### How close is the answer?

**Monte Carlo error.** The 256 draws are a random sample from the distribution, so a lifted result differs from the exact one by a random amount.
Four times as many draws halve the typical error, and `f.using(max_evaluations=4000)` asks for 4,000 draws.

### How do I get a number out, and how was it computed?

**Operations** turn a distribution into the summaries you report, such as `mean` and `quantile`.
An operation uses a formula when one exists, as one does for the mean of a normal distribution, and an algorithm otherwise.

**Provenance** is the record that every result keeps of how it was computed.
Printing `result.provenance` lists each step and marks it as exact or approximate, so you can see which numbers came from random draws.

### Can I get the same answer again?

**Seeds.** Random draws start from a seed.
Run a computation inside `with workflow_run(seed=0):`, and every draw in the block is the same each time you run it.

The [first page of the user guide](index.md) applies the same concepts to the Challenger data.

## Example: an uncertain braking distance

A car travels at 25 m/s (90 km/h) on dry asphalt, whose friction coefficient is $\mu = 0.7$.
Its braking distance is $v^2 / (2 \mu g)$, which is 45.5 m at that speed.
Suppose the speed is known only to within about 4 m/s.
The braking distance is then uncertain too, and its mean is larger than the 45.5 m that the best guess of the speed gives.
The sections below compute its distribution and its mean, one concept at a time.

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab's preinstalled PyMC 5 does not work with ArviZ 1.x, so the install upgrades PyMC.
    %pip install --quiet "probpipe-core[pymc] @ git+https://github.com/TARPS-group/prob-pipe.git@main"

In [2]:
import jax
import jax.numpy as jnp

from probpipe import Normal, function, mean, quantile, workflow_run

### Values and functions

`stopping_distance` computes the braking distance, in meters, for one speed and one friction coefficient, in plain JAX.
`@function` makes it a ProbPipe function, so it accepts values and distributions as its inputs:

In [3]:
@function
def stopping_distance(speed: jax.Array, friction: jax.Array) -> jax.Array:
    return speed**2 / (2 * friction * 9.81)


known_distance = stopping_distance(25.0, 0.7)
print("braking distance at 25 m/s, in m:", known_distance.raw)
print("type of the result:", type(known_distance).__name__)

braking distance at 25 m/s, in m: 45.5075
type of the result: NumericArray


Called with numbers, the function returns a value, here a `NumericArray`.

### Distributions and lifting

A normal distribution with mean 25 and standard deviation 4 describes the uncertain speed.
The first argument of `Normal`, `"speed"`, is its output name, which names the quantity that a draw represents.
Passing the distribution in place of the number lifts the function, and the result is the distribution of the braking distance:

In [4]:
speed = Normal("speed", 25.0, 4.0)

with workflow_run(seed=0):
    distance = stopping_distance(speed, 0.7)

print("type of the result:", type(distance).__name__)
print("number of draws:", distance.num_atoms)
print("mean braking distance, in m:", mean(distance).raw)
print("5% and 95% quantiles, in m:", quantile(distance, jnp.array([0.05, 0.95])).raw)

type of the result: EmpiricalDistribution
number of draws: 256
mean braking distance, in m: 45.847313


5% and 95% quantiles, in m: [23.493431 70.27566 ]


The function body is unchanged, and it still computes one distance from one speed.
ProbPipe drew 256 speeds from the distribution, evaluated the body at each draw, and returned an `EmpiricalDistribution`, which is a distribution on the 256 distances.
The `workflow_run` block sets the random seed, which [Reproducible draws](#reproducible-draws) explains.

### The same computation by hand

Lifting is a Monte Carlo computation, and a few lines of JAX carry out the same computation:

In [5]:
key = jax.random.key(0)
speed_draws = 25.0 + 4.0 * jax.random.normal(key, (256,))
distance_draws = speed_draws**2 / (2 * 0.7 * 9.81)
print("mean braking distance by hand, in m:", distance_draws.mean())

mean braking distance by hand, in m: 47.03794


The hand-written version draws a different set of 256 speeds, so its mean differs from ProbPipe's.
It also asks more of you, in three ways:

1. **The function:** the formula is rewritten to act on an array of draws, and a function that is not elementwise needs `jax.vmap` or a loop.
2. **The random key:** you create the key and pass it to every draw.
3. **The record:** nothing records that the mean came from 256 random draws.

ProbPipe keeps the function for one value, manages the key, and records how each result was computed.

### How accurate is a lifted result?

The exact mean has a formula here, because the mean of the squared speed is $25^2 + 4^2 = 641$.
Two other answers compare with it.
The plug-in answer evaluates the function at the mean speed.
The lifted mean with 4,000 draws uses `stopping_distance.using(max_evaluations=4000)`, which is a copy of the function that evaluates 4,000 draws:

In [6]:
exact_mean = (25.0**2 + 4.0**2) / (2 * 0.7 * 9.81)
plug_in = stopping_distance(mean(speed), 0.7)

with workflow_run(seed=0):
    distance_4000 = stopping_distance.using(max_evaluations=4000)(speed, 0.7)

print("exact mean, in m:", round(exact_mean, 2))
print("plug-in at the mean speed, in m:", plug_in.raw)
print("lifted mean with 256 draws, in m:", mean(distance).raw)
print("lifted mean with 4000 draws, in m:", mean(distance_4000).raw)

exact mean, in m: 46.67
plug-in at the mean speed, in m: 45.5075
lifted mean with 256 draws, in m: 45.847313
lifted mean with 4000 draws, in m: 46.71466


The plug-in answer is 1.2 m short of the exact mean, and more draws do not change it, because the distance grows faster than linearly in the speed.
The lifted mean differs from the exact mean by Monte Carlo error.
Here 256 draws give a mean 0.8 m short of the exact mean, and 4,000 draws give a mean within 0.1 m of it.
[Propagating uncertainty through functions](03_functions_and_lifting.ipynb) covers the number of draws and the other settings of a lifted call.

### Operations and provenance

The operation `mean` uses a formula for a normal distribution, and the provenance of each result shows which steps were exact:

In [7]:
print("provenance of the mean speed:")
print(mean(speed).provenance)
print("provenance of the mean braking distance:")
print(mean(distance).provenance)

provenance of the mean speed:
1. mean(Normal, given)  route closed_form, exact
provenance of the mean braking distance:
1. stopping_distance = broadcast(stopping_distance, Normal, friction)  route sampling_lift, approximate
2. mean(1, given)                                    route closed_form, exact


A normal distribution's mean has a formula, so the mean speed is exact.
The mean braking distance is the average of 256 draws, which a formula also gives.
That result is approximate all the same, because its first step, the lifted call, is approximate.
[Tracking and reproducing results](09_how_results_are_computed.ipynb) explains the other entries of a provenance.

### Reproducible draws

Running the lifted call again under the same seed gives the same draws, and a different seed gives different draws:

In [8]:
with workflow_run(seed=0):
    repeated = stopping_distance(speed, 0.7)
with workflow_run(seed=1):
    reseeded = stopping_distance(speed, 0.7)

print("mean with seed 0, first run, in m:", mean(distance).raw)
print("mean with seed 0, second run, in m:", mean(repeated).raw)
print("mean with seed 1, in m:", mean(reseeded).raw)

mean with seed 0, first run, in m: 45.847313
mean with seed 0, second run, in m: 45.847313
mean with seed 1, in m: 47.159645


## Where to go next

- [Distributions](02_distributions.ipynb): construct distributions, draw values, and compute summaries.
- [Propagating uncertainty through functions](03_functions_and_lifting.ipynb): lift your own functions, set the number of draws, and declare the inputs and the output that a function accepts.
- [Your data: values, records, and batches](04_values_and_records.ipynb): pass data into computations and read the results.

## See also

- [Values and records](../api/values.md): the API reference for `NumericArray` and the other values.
- [Functions](../api/functions.md): `function` and the rules for calling a function on values, distributions, and batches.
- [Operations](../api/operations.md): `mean`, `quantile`, and the other operations.
- [Workflows and reproducibility](../api/workflows.md): `workflow_run`.
- [Tutorial 1](../tutorials/01_first_analysis.ipynb) follows a complete analysis from a prior to predictions.